In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [4]:
df = spark.read.json("cwiczenia/RTA/transactions_10k.jsonl")



In [5]:
print(f"Liczba rekordów: {df.count()}")
df.printSchema()

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [6]:
df.show(10, truncate=False)

+------+-----------+--------+-------------------+-------+-------+
|amount|category   |store   |timestamp          |tx_id  |user_id|
+------+-----------+--------+-------------------+-------+-------+
|312.32|elektronika|Warszawa|2026-04-12 08:25:07|TX00001|u48    |
|79.57 |książki    |Warszawa|2026-04-12 08:05:43|TX00002|u15    |
|126.17|odzież     |Warszawa|2026-04-12 09:15:30|TX00003|u18    |
|34.08 |odzież     |Warszawa|2026-04-12 10:05:39|TX00004|u10    |
|428.88|żywność    |Kraków  |2026-04-12 09:04:36|TX00005|u17    |
|345.21|książki    |Warszawa|2026-04-12 09:36:31|TX00006|u25    |
|376.42|żywność    |Warszawa|2026-04-12 10:06:49|TX00007|u15    |
|85.36 |elektronika|Gdańsk  |2026-04-12 09:08:25|TX00008|u24    |
|66.26 |żywność    |Kraków  |2026-04-12 10:06:19|TX00009|u05    |
|660.41|odzież     |Kraków  |2026-04-12 08:29:24|TX00010|u41    |
+------+-----------+--------+-------------------+-------+-------+
only showing top 10 rows



In [13]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  # timestamp powinien być teraz 'timestamp (nullable = true)'

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [7]:
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)
store_summary.show()

+--------+---------+----------+-----------+
|   store|liczba_tx|  suma_PLN|srednia_PLN|
+--------+---------+----------+-----------+
|  Gdańsk|     2498|1021266.35|     408.83|
|  Kraków|     2522|1025896.95|     406.78|
|Warszawa|     2424| 961642.24|     396.72|
| Wrocław|     2556|1002739.21|     392.31|
+--------+---------+----------+-----------+



In [8]:
from pyspark.sql.functions import sum as _sum, min as _min, max as _max, round as _round

df.groupBy("category") \
    .agg(
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(_min("amount"), 2).alias("min_PLN"),
        _round(_max("amount"), 2).alias("max_PLN"),
    ) \
    .orderBy("category") \
    .show()

+-----------+----------+-------+-------+
|   category|  suma_PLN|min_PLN|max_PLN|
+-----------+----------+-------+-------+
|elektronika|1520770.69|    9.0| 9999.0|
|    książki| 851382.08|    5.0|9107.25|
|     odzież| 849877.55|    5.0|9696.63|
|    żywność| 789514.43|    5.0|6916.92|
+-----------+----------+-------+-------+



In [9]:
from pyspark.sql.functions import window

hourly = (
    df.groupBy(window("timestamp", "1 hour"))    # okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

+------------------------------------------+---------+----------+
|window                                    |liczba_tx|suma_PLN  |
+------------------------------------------+---------+----------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|3150     |1241911.3 |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|4661     |1896230.21|
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|2189     |873403.24 |
+------------------------------------------+---------+----------+



In [14]:
(
    hourly
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .show(truncate=False)
)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
+-------------------+-------------------+---------+----------+



In [16]:
from pyspark.sql.functions import window, count, sum as _sum, round as _round, col

store_30min = (
    df.groupBy(window("timestamp", "30 minutes"), "store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window", "store")
)

store_30min.show(truncate=False)

+------------------------------------------+--------+---------+---------+
|window                                    |store   |liczba_tx|suma_PLN |
+------------------------------------------+--------+---------+---------+
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Gdańsk  |252      |93391.22 |
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Kraków  |289      |117786.42|
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Warszawa|275      |88441.58 |
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Wrocław |296      |111540.59|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Gdańsk  |514      |209187.85|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Kraków  |532      |223541.41|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Warszawa|490      |182435.06|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Wrocław |502      |215587.17|
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|Gdańsk  |619      |253364.95|
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|Kraków  |590      |224358.03|
|{2026-04-12 09:00:00, 2026-04-12 09:3

In [17]:
from pyspark.sql.functions import window, sum as _sum, round as _round, col, desc

krakow_best_hour = (
    df.filter(col("store") == "Kraków")
    .groupBy(window("timestamp", "1 hour"))
    .agg(
        _round(_sum("amount"), 2).alias("suma_PLN")
    )
    .orderBy(desc("suma_PLN"))
)

krakow_best_hour.show(truncate=False)

+------------------------------------------+---------+
|window                                    |suma_PLN |
+------------------------------------------+---------+
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|483309.86|
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|341327.83|
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|201259.26|
+------------------------------------------+---------+



In [15]:
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))  # szerokość 1h, krok 30min
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od")
)
sliding.show(truncate=False)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 07:30:00|2026-04-12 08:30:00|1112     |411159.81 |
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 08:30:00|2026-04-12 09:30:00|4443     |1753033.6 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 09:30:00|2026-04-12 10:30:00|3696     |1557641.39|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
|2026-04-12 10:30:00|2026-04-12 11:30:00|749      |289709.95 |
+-------------------+-------------------+---------+----------+



In [15]:
tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)
sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)
print(f"Tumbling (1h):          {tumbling_rows} okien")
print(f"Sliding  (1h / 30min):  {sliding_rows} okien")

Tumbling (1h):          3 okien
Sliding  (1h / 30min):  7 okien


In [ ]:
# Dlaczego sliding ma więcej wierszy?
# Sliding ma więcej wierszy, bo okna się nakładają i startują częściej (co 30 min).
# Ta sama transakcja może być policzona w kilku oknach, a w tumbling tylko w jednym.

In [ ]:
# Odpowiedz na pytania w komentarzach:

# 1. Ile transakcji jest w oknie 09:00–10:00?
#    Sprawdź w wyniku zadania 3.1.
#    ODPOWIEDŹ: 4661

# 2. Jaka jest różnica między groupBy("store") a groupBy(window(...), "store")?
#    ODPOWIEDŹ: groupBy("store") grupuje wszystkie transakcje tylko według sklepu,
#    bez uwzględniania czasu. Dostajemy jeden łączny wynik dla każdego sklepu.
#    groupBy(window(...), "store") grupuje jednocześnie po czasie i po sklepie,
#    więc dla każdego sklepu powstaje osobny wynik w każdym oknie czasowym.

# 3. W oknie sliding 1h/30min — ile okien zawiera transakcje z godziny 09:30?
#    Wskazówka: narysuj oś czasu.
#    ODPOWIEDŹ: 2 okna

In [21]:
# W której godzinie Gdańśk najniższą średnią kwotę transakcji.
from pyspark.sql.functions import window, col, avg, round as _round, asc

gdansk_lowest_avg = (
    df.filter(col("store") == "Gdańsk")
    .groupBy(window("timestamp", "1 hour"))
    .agg(
        _round(avg("amount"), 2).alias("srednia_PLN")
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        col("srednia_PLN")
    )
    .orderBy(asc("srednia_PLN"))
)

gdansk_lowest_avg.show(1, truncate=False)

+-------------------+-------------------+-----------+
|od                 |do                 |srednia_PLN|
+-------------------+-------------------+-----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|395.01     |
+-------------------+-------------------+-----------+
only showing top 1 row



In [24]:
#Policz ile transakcji per kategoria było w oknie 09:00–09:30.
from pyspark.sql.functions import window, count, col

category_0930 = (
    df.groupBy(window("timestamp", "30 minutes"), "category")
    .agg(
        count("tx_id").alias("liczba_tx")
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        col("category"),
        col("liczba_tx")
    )
    .filter(
        (col("od") == "2026-04-12 09:00:00") &
        (col("do") == "2026-04-12 09:30:00")
    )
    .orderBy("category")
)

category_0930.show(truncate=False)

+-------------------+-------------------+-----------+---------+
|od                 |do                 |category   |liczba_tx|
+-------------------+-------------------+-----------+---------+
|2026-04-12 09:00:00|2026-04-12 09:30:00|elektronika|611      |
|2026-04-12 09:00:00|2026-04-12 09:30:00|książki    |622      |
|2026-04-12 09:00:00|2026-04-12 09:30:00|odzież     |605      |
|2026-04-12 09:00:00|2026-04-12 09:30:00|żywność    |567      |
+-------------------+-------------------+-----------+---------+



In [25]:
from pyspark.sql.functions import window, count, col

window_15min = (
    df.groupBy(window("timestamp", "15 minutes"))
    .agg(
        count("tx_id").alias("liczba_tx")
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        col("liczba_tx")
    )
    .orderBy("od")
)

window_15min.show(truncate=False)
# peak był w przedziale czasowym 9:15 - 9:30

+-------------------+-------------------+---------+
|od                 |do                 |liczba_tx|
+-------------------+-------------------+---------+
|2026-04-12 08:00:00|2026-04-12 08:15:00|468      |
|2026-04-12 08:15:00|2026-04-12 08:30:00|644      |
|2026-04-12 08:30:00|2026-04-12 08:45:00|899      |
|2026-04-12 08:45:00|2026-04-12 09:00:00|1139     |
|2026-04-12 09:00:00|2026-04-12 09:15:00|1171     |
|2026-04-12 09:15:00|2026-04-12 09:30:00|1234     |
|2026-04-12 09:30:00|2026-04-12 09:45:00|1156     |
|2026-04-12 09:45:00|2026-04-12 10:00:00|1100     |
|2026-04-12 10:00:00|2026-04-12 10:15:00|858      |
|2026-04-12 10:15:00|2026-04-12 10:30:00|582      |
|2026-04-12 10:30:00|2026-04-12 10:45:00|443      |
|2026-04-12 10:45:00|2026-04-12 11:00:00|306      |
+-------------------+-------------------+---------+

